# 🧹 Data Cleaning & Feature Engineering

## 🎯 Objective

Clean and transform the healthcare appointment dataset to improve data quality, standardize fields, and create analytical features for SQL analysis, data modeling, and dashboard development.

## 🔍 Loading Necessary Libraries

Import required Python libraries for data cleaning and feature engineering.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np

print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)

Pandas Version: 2.0.3
NumPy Version: 1.26.4


## 🔍 Loading the Dataset

Load the raw healthcare appointment dataset into a Pandas DataFrame for cleaning and transformation.

In [2]:
# Load raw dataset

df = pd.read_csv("../data/raw/dataset.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


## 🔍 Column Standardization

Review and standardize column names to improve readability, consistency, and maintainability.

In [3]:
# Display current column names

df.columns.tolist()

['PatientId',
 'AppointmentID',
 'Gender',
 'ScheduledDay',
 'AppointmentDay',
 'Age',
 'Neighbourhood',
 'Scholarship',
 'Hipertension',
 'Diabetes',
 'Alcoholism',
 'Handcap',
 'SMS_received',
 'No-show']

In [4]:
# Rename columns for consistency

df = df.rename(
    columns={
        "Hipertension": "Hypertension"
    }
)

In [5]:
# Verify updated column names

df.columns.tolist()

['PatientId',
 'AppointmentID',
 'Gender',
 'ScheduledDay',
 'AppointmentDay',
 'Age',
 'Neighbourhood',
 'Scholarship',
 'Hypertension',
 'Diabetes',
 'Alcoholism',
 'Handcap',
 'SMS_received',
 'No-show']

## 🔍 Data Type Corrections

Review and standardize data types to support data quality, feature engineering, and analytical modeling.

In [6]:
# Display current data types

df.dtypes

PatientId         float64
AppointmentID       int64
Gender             object
ScheduledDay       object
AppointmentDay     object
Age                 int64
Neighbourhood      object
Scholarship         int64
Hypertension        int64
Diabetes            int64
Alcoholism          int64
Handcap             int64
SMS_received        int64
No-show            object
dtype: object

In [7]:
# Convert PatientId to integer

df["PatientId"] = df["PatientId"].astype("int64")

In [8]:
# Convert date columns to datetime

df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])

In [9]:
# Verify updated data types

df.dtypes

PatientId                       int64
AppointmentID                   int64
Gender                         object
ScheduledDay      datetime64[ns, UTC]
AppointmentDay    datetime64[ns, UTC]
Age                             int64
Neighbourhood                  object
Scholarship                     int64
Hypertension                    int64
Diabetes                        int64
Alcoholism                      int64
Handcap                         int64
SMS_received                    int64
No-show                        object
dtype: object

## 🔍 Invalid Age Handling

Review and address invalid age values to improve data quality and ensure reliable demographic analysis.

In [10]:
# Count records with invalid age values

invalid_age_count = (df["Age"] < 0).sum()

print(f"Invalid Age Records: {invalid_age_count}")

Invalid Age Records: 1


In [11]:
# Display records with invalid age values

df[df["Age"] < 0]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
99832,465943158731293,5775010,F,2016-06-06 08:58:13+00:00,2016-06-06 00:00:00+00:00,-1,ROMÃO,0,0,0,0,0,0,No


In [12]:
# Remove records with invalid age values

df = df[df["Age"] >= 0]

In [13]:
# Verify invalid age records have been removed

invalid_age_count = (df["Age"] < 0).sum()

print(f"Invalid Age Records: {invalid_age_count}")

Invalid Age Records: 0


In [14]:
# Display updated dataset dimensions

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 110,526
Columns: 14


## 🔍 Negative Waiting Time Analysis

Review records where the appointment date occurs before the scheduling date and determine the appropriate remediation strategy.

In [15]:
# Create corrected waiting days feature

df["WaitingDays"] = (
    df["AppointmentDay"].dt.normalize()
    - df["ScheduledDay"].dt.normalize()
).dt.days

In [16]:
# Count negative waiting time records

negative_waiting_count = (df["WaitingDays"] < 0).sum()

print(f"Negative Waiting Time Records: {negative_waiting_count}")

Negative Waiting Time Records: 5


In [17]:
# Display negative waiting time records

df[df["WaitingDays"] < 0][
    [
        "PatientId",
        "AppointmentID",
        "ScheduledDay",
        "AppointmentDay",
        "WaitingDays"
    ]
]

,PatientId,AppointmentID,ScheduledDay,AppointmentDay,WaitingDays
27033,7839272661752,5679978,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,-1
55226,7896293967868,5715660,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,-1
64175,24252258389979,5664962,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,-1
71533,998231581612122,5686628,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,-6
72362,3787481966821,5655637,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,-1


## 🔍 Negative Waiting Time Remediation

Records with negative waiting times indicate that the appointment date occurred before the scheduling date. Since these records violate business logic and represent a negligible portion of the dataset, they will be removed from further analysis.

In [18]:
# Remove records with negative waiting times

df = df[df["WaitingDays"] >= 0]

In [19]:
# Validate removal

negative_waiting_count = (df["WaitingDays"] < 0).sum()

print(f"Negative Waiting Time Records: {negative_waiting_count}")

Negative Waiting Time Records: 0


In [20]:
# Display updated dataset dimensions

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 110,521
Columns: 15


## ⚙️ Feature Engineering

Create additional analytical features to support SQL analysis, reporting, and dashboard development.

### 👥 Age Group Creation

Categorize patients into age segments to support demographic analysis and reporting.

In [25]:
# Create age groups

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 17, 35, 55, 115],
    labels=[
        "Child",
        "Young Adult",
        "Adult",
        "Senior"
    ],
    include_lowest=True
)

In [27]:
# Check for missing age groups

df["AgeGroup"].isna().sum()

0

### 📊 Age Group Validation

Review the distribution of patients across age segments after age group classification.

In [28]:
# Review age group distribution

df["AgeGroup"].value_counts()

AgeGroup
Adult          30018
Senior         27503
Child          27378
Young Adult    25622
Name: count, dtype: int64

### 📅 Attendance Status Creation

Create a business-friendly attendance indicator derived from the original `No-show` field.

In [29]:
# Create attendance status feature

df["AttendanceStatus"] = df["No-show"].map({
    "No": "Attended",
    "Yes": "Missed"
})

In [30]:
# Review attendance status distribution

df["AttendanceStatus"].value_counts()

AttendanceStatus
Attended    88207
Missed      22314
Name: count, dtype: int64

In [31]:
# Check for missing values

df["AttendanceStatus"].isna().sum()

0

## 💾 Export Cleaned Dataset

Export the cleaned and transformed dataset for downstream analytics and reporting.

In [35]:
# Reset index after removing records

df = df.reset_index(drop=True)

# Export cleaned dataset

df.to_csv(
    "../data/processed/appointments_cleaned.csv",
    index=False
)

print("Cleaned dataset exported successfully!")

Cleaned dataset exported successfully!


In [36]:
# Verify dataset dimensions

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 110,521
Columns: 17


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110521 entries, 0 to 110520
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   PatientId         110521 non-null  int64              
 1   AppointmentID     110521 non-null  int64              
 2   Gender            110521 non-null  object             
 3   ScheduledDay      110521 non-null  datetime64[ns, UTC]
 4   AppointmentDay    110521 non-null  datetime64[ns, UTC]
 5   Age               110521 non-null  int64              
 6   Neighbourhood     110521 non-null  object             
 7   Scholarship       110521 non-null  int64              
 8   Hypertension      110521 non-null  int64              
 9   Diabetes          110521 non-null  int64              
 10  Alcoholism        110521 non-null  int64              
 11  Handcap           110521 non-null  int64              
 12  SMS_received      110521 non-null  int64    

## ✅ Data Cleaning Summary

The healthcare appointment dataset was successfully cleaned, validated, and enhanced for downstream analytics.

---

### 🏷️ Column Standardization

- Renamed `Hipertension` to `Hypertension`

---

### 🔄 Data Type Corrections

- Converted `PatientId` from `float64` to `int64`
- Converted `ScheduledDay` to datetime format
- Converted `AppointmentDay` to datetime format

---

### ⚠️ Data Quality Remediation

- Removed 1 record with an invalid age value (`Age = -1`)

- Removed 5 records with negative waiting times

- Reset dataset index after record removal

---

### ⚙️ Feature Engineering

Created the following analytical features:

- `WaitingDays`
- `AgeGroup`
- `AttendanceStatus`

---

### 📊 Final Dataset Statistics

| Metric | Value |
|----------|----------|
| Rows | 110,521 |
| Columns | 17 |

---

### 🚀 Outcome

The dataset is now clean, validated, and analytics-ready for:

- PostgreSQL data loading
- SQL analysis
- Exploratory Data Analysis (EDA)
- Power BI dashboard development